### 9. End-to-End Development

#### 1. Objetivo

El objetivo de esta fase es validar la integración completa de los componentes desarrollados previamente.

La función `analyze_news()` recibe el título y el contenido de una noticia y coordina secuencialmente la detección de idioma, extracción de claims, investigación web, recuperación de evidencias, verificación factual y síntesis del veredicto final.

En una primera prueba se utiliza una noticia en español, por lo que no interviene la rama auxiliar de Machine Learning.

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [53]:
from dotenv import load_dotenv
from openai import OpenAI
from tavily import TavilyClient

load_dotenv("../.env")

client = OpenAI()
tavily_client = TavilyClient()

In [48]:
import importlib
import src.backend as backend

importlib.reload(backend)

<module 'src.backend' from 'C:\\Users\\natal\\OneDrive\\Documentos\\UCM - Big data, ML and IA\\MODULOS\\TFM\\TFM_Natalia_De_Oliveira_AgenteFakeNews\\src\\backend.py'>

In [40]:
from src.backend import (analyze_news,
                        NewsAnalysisResult,
                        ClaimAnalysisDetail,
                        VerifiedEvidence,
)
from src.live_retrieval import load_embedding_model

In [ ]:
embedding_model = load_embedding_model()

c:\Users\natal\anaconda3\envs\tfm_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3165.96it/s]


In [4]:
title = "La Unión Europea prohibirá completamente los pagos en efectivo en 2027"

body = """
La Unión Europea eliminará por completo la posibilidad de realizar pagos
en efectivo a partir de 2027. La medida entrará en vigor en julio de ese año
y obligará a ciudadanos y empresas a utilizar exclusivamente medios de pago
digitales.
"""

In [6]:
result = analyze_news(
    title=title,
    body=body,
    client=client,
    tavily_client=tavily_client,
    embedding_model=embedding_model,
)

result

FinalVerdict(verdict='REFUTED', explanation='El artículo queda refutado. La evidencia suficiente indica que la norma de la UE aplicable desde el 10 de julio de 2027 no elimina los pagos en efectivo ni obliga a ciudadanos y empresas a usar exclusivamente medios digitales. Establece un límite de 10.000 euros para determinadas transacciones en efectivo, principalmente comerciales, con excepciones y la posibilidad de normas nacionales más estrictas. Aunque la fecha de entrada en vigor mencionada es correcta, la descripción central de la medida como una prohibición total del efectivo es falsa.')

### 2. Primera prueba end-to-end en español

Se ejecuta la función `analyze_news()` utilizando una noticia en español.

En esta prueba no interviene la rama auxiliar de Machine Learning, por lo que el veredicto final se obtiene exclusivamente a partir del flujo factual:

`detección de idioma → extracción de claims → Research Agent → Live RAG → Evidence Verifier → Verdict Synthesizer`.

El sistema devuelve un veredicto final `REFUTED`.

La explicación final indica que la normativa europea aplicable desde el 10 de julio de 2027 no elimina completamente los pagos en efectivo ni obliga al uso exclusivo de medios digitales. En su lugar, establece un límite de 10.000 euros para determinadas operaciones en efectivo, manteniendo además excepciones y la posibilidad de que los Estados miembros apliquen límites inferiores.

Aunque la referencia temporal incluida en la noticia es correcta, la afirmación principal sobre una prohibición total del efectivo resulta refutada.

Esta prueba confirma que los distintos componentes desarrollados de forma independiente pueden integrarse correctamente en un flujo end-to-end.

### 3. Salida estructurada del backend

Tras validar el funcionamiento end-to-end del pipeline, se amplía la salida de `analyze_news()` para conservar no solo el veredicto final, sino también información intermedia relevante del proceso.

La nueva estructura incluye:

- el idioma detectado;
- las claims extraídas por el Claim Analyzer;
- el resultado de verificación de cada claim;
- la señal auxiliar del modelo de Machine Learning, cuando exista;
- el veredicto final sintetizado.

El objetivo es mantener trazabilidad sobre las distintas etapas del análisis y facilitar posteriormente la integración con LangGraph o una interfaz de usuario.

In [11]:
result = analyze_news(
    title=title,
    body=body,
    client=client,
    tavily_client=tavily_client,
    embedding_model=embedding_model,
)

result

NewsAnalysisResult(language='es', claims=[ClaimItem(id=1, claim='La Unión Europea prohibirá completamente los pagos en efectivo a partir de julio de 2027.', entities=['Unión Europea'], date_reference='julio de 2027'), ClaimItem(id=2, claim='A partir de julio de 2027, la medida obligará a ciudadanos y empresas de la Unión Europea a utilizar exclusivamente medios de pago digitales.', entities=['Unión Europea'], date_reference='julio de 2027')], claim_verifications=[ClaimVerificationSummary(claim='La Unión Europea prohibirá completamente los pagos en efectivo a partir de julio de 2027.', verdict='REFUTED', evidence_sufficient=True, explanation='The evidence establishes that an EU anti-money-laundering regulation will apply from 10 July 2027, but characterizes its effect as a common limit or threshold for cash payments, not a complete prohibition. Therefore, the claim is false in its essential assertion that all cash payments will be banned.'), ClaimVerificationSummary(claim='A partir de j

In [12]:
print("LANGUAGE:", result.language)
print("=" * 100)

print("CLAIMS:")
for claim in result.claims:
    print(claim)
    print("-" * 80)

print("=" * 100)

print("CLAIM VERIFICATIONS:")
for verification in result.claim_verifications:
    print(verification)
    print("-" * 80)

print("=" * 100)

print("ML SIGNAL:", result.ml_signal)

print("=" * 100)

print("FINAL VERDICT:")
print(result.final_verdict)

LANGUAGE: es
CLAIMS:
id=1 claim='La Unión Europea prohibirá completamente los pagos en efectivo a partir de julio de 2027.' entities=['Unión Europea'] date_reference='julio de 2027'
--------------------------------------------------------------------------------
id=2 claim='A partir de julio de 2027, la medida obligará a ciudadanos y empresas de la Unión Europea a utilizar exclusivamente medios de pago digitales.' entities=['Unión Europea'] date_reference='julio de 2027'
--------------------------------------------------------------------------------
CLAIM VERIFICATIONS:
claim='La Unión Europea prohibirá completamente los pagos en efectivo a partir de julio de 2027.' verdict='REFUTED' evidence_sufficient=True explanation='The evidence establishes that an EU anti-money-laundering regulation will apply from 10 July 2027, but characterizes its effect as a common limit or threshold for cash payments, not a complete prohibition. Therefore, the claim is false in its essential assertion that 

#### 3.1. Resultado

La ejecución mantiene el veredicto final `REFUTED` y conserva correctamente la información intermedia del análisis.

En la prueba se detecta el idioma español, se extraen dos claims verificables y ambas son clasificadas como `REFUTED` con evidencia suficiente. Al tratarse de una noticia en español, la señal de Machine Learning permanece como `None`.

La prueba confirma que el backend puede devolver una salida estructurada sin modificar la lógica factual previamente validada.

### 4. Trazabilidad de evidencias por claim

La salida estructurada del backend se amplía para conservar también las evidencias utilizadas durante la verificación de cada claim.

Hasta este punto, el resultado almacenaba el veredicto y la explicación generados por el Evidence Verifier, pero no mantenía explícitamente la relación entre cada claim y los fragmentos documentales recuperados por el Live RAG.

Para mejorar la trazabilidad del sistema se incorporan dos nuevas estructuras:

- `VerifiedEvidence`, que combina la información recuperada por el RAG con la valoración realizada por el Evidence Verifier;
- `ClaimAnalysisDetail`, que agrupa la claim original, su resultado de verificación y las evidencias utilizadas.

Para cada evidencia se conservan, entre otros elementos, la URL, el texto recuperado, el score de similitud semántica, la relación `SUPPORTS`, `REFUTES` o `NEUTRAL`, y la justificación proporcionada por el Evidence Verifier.

Esta información permite reconstruir el recorrido completo desde las fuentes recuperadas hasta el veredicto obtenido para cada claim.

### 4.1. Validación de la trazabilidad

Se vuelve a ejecutar la misma noticia de prueba para comprobar que la nueva estructura conserva correctamente las evidencias utilizadas para cada claim y su valoración correspondiente.

El objetivo de esta prueba no es volver a evaluar la calidad del veredicto final, sino verificar que la información generada por el Live RAG y el Evidence Verifier queda correctamente asociada dentro de la salida end-to-end.

In [18]:
result = analyze_news(
    title=title,
    body=body,
    client=client,
    tavily_client=tavily_client,
    embedding_model=embedding_model,
)

result

NewsAnalysisResult(language='es', claims=[ClaimItem(id=1, claim='La Unión Europea eliminará por completo la posibilidad de realizar pagos en efectivo a partir de julio de 2027.', entities=['Unión Europea'], date_reference='julio de 2027'), ClaimItem(id=2, claim='A partir de julio de 2027, la Unión Europea obligará a ciudadanos y empresas a utilizar exclusivamente medios de pago digitales.', entities=['Unión Europea'], date_reference='julio de 2027')], claim_verifications=[ClaimVerificationSummary(claim='La Unión Europea eliminará por completo la posibilidad de realizar pagos en efectivo a partir de julio de 2027.', verdict='REFUTED', evidence_sufficient=True, explanation='The evidence consistently describes a targeted EU anti-money-laundering measure effective on 10 July 2027: a EUR 10,000 ceiling for specified commercial cash transactions, together with identification requirements and exceptions. It does not eliminate cash payments altogether; evidence explicitly characterizes the mea

In [21]:
for detail in result.claim_details:

    print("=" * 100)

    print("CLAIM:")
    print(detail.claim.claim)

    print("\nVERDICT:")
    print(detail.verification.verdict)

    print("\nEVIDENCE SUFFICIENT:")
    print(detail.verification.evidence_sufficient)

    print("\nEXPLANATION:")
    print(detail.verification.explanation)

    print("\nEVIDENCES:")

    for evidence in detail.evidences:

        print("-" * 80)

        print("EVIDENCE ID:")
        print(evidence.evidence_id)

        print("\nTITLE:")
        print(evidence.title)

        print("\nURL:")
        print(evidence.url)

        print("\nRAG SCORE:")
        print(evidence.score)

        print("\nRELATION:")
        print(evidence.relation)

        print("\nREASON:")
        print(evidence.reason)

        print("\nTEXT:")
        print(evidence.text[:500])

print("=" * 100)
print("FINAL VERDICT:")
print(result.final_verdict.verdict)

print("\nFINAL EXPLANATION:")
print(result.final_verdict.explanation)

CLAIM:
La Unión Europea eliminará por completo la posibilidad de realizar pagos en efectivo a partir de julio de 2027.

VERDICT:
REFUTED

EVIDENCE SUFFICIENT:
True

EXPLANATION:
The evidence consistently describes a targeted EU anti-money-laundering measure effective on 10 July 2027: a EUR 10,000 ceiling for specified commercial cash transactions, together with identification requirements and exceptions. It does not eliminate cash payments altogether; evidence explicitly characterizes the measure as a limit on large cash payments. Therefore, the claim that the EU will completely remove the possibility of making cash payments from July 2027 is refuted.

EVIDENCES:
--------------------------------------------------------------------------------
EVIDENCE ID:
1

TITLE:
Guerra contra la economía sumergida: la UE pone cerco a ...

URL:
https://es.euronews.com/my-europe/2026/06/10/ue-pone-cerco-grandes-pagos-efectivo

RAG SCORE:
0.9165650606155396

RELATION:
REFUTES

REASON:
It states that fr

### 5. Integración del clasificador ML en el flujo end-to-end

Una vez validado el pipeline factual y la trazabilidad de evidencias, se incorpora al backend la señal auxiliar generada por el clasificador de Machine Learning para noticias en inglés.

El modelo ML se utiliza únicamente como una señal complementaria basada en las características textuales del artículo completo.

La verificación factual mantiene prioridad sobre esta predicción, por lo que la señal ML no puede sustituir ni invalidar resultados respaldados por evidencia suficiente.

Para noticias en español, la rama ML no se ejecuta y `ml_signal` permanece como `None`.

#### 5.1. Validación end-to-end con una noticia en inglés

Se ejecuta una noticia en inglés para comprobar que el backend:

- detecta correctamente el idioma;
- ejecuta la rama factual completa;
- obtiene la predicción del clasificador ML;
- incorpora dicha señal al Verdict Synthesizer;
- conserva el resultado ML dentro de la salida estructurada `NewsAnalysisResult`.

In [27]:
title_en = (
    "The European Union will completely ban cash payments from 2027"
)

body_en = """
The European Union will completely eliminate cash payments starting
in July 2027. Citizens and businesses across the EU will be required
to use digital payment methods exclusively.
"""

In [28]:
result_en = analyze_news(
    title=title_en,
    body=body_en,
    client=client,
    tavily_client=tavily_client,
    embedding_model=embedding_model,
)

result_en

NewsAnalysisResult(language='en', claims=[ClaimItem(id=1, claim='The European Union will completely eliminate cash payments starting in July 2027.', entities=['European Union'], date_reference='July 2027'), ClaimItem(id=2, claim='Starting in July 2027, citizens and businesses across the European Union will be required to use digital payment methods exclusively.', entities=['European Union'], date_reference='July 2027')], claim_verifications=[ClaimVerificationSummary(claim='The European Union will completely eliminate cash payments starting in July 2027.', verdict='REFUTED', evidence_sufficient=True, explanation="The evidence supports that EU anti-money-laundering rules become applicable on 10 July 2027 and impose a €10,000 ceiling on certain cash payments. However, it consistently shows that cash is not completely eliminated: payments below the threshold remain possible, the rule concerns commercial or professional transactions, and specified exceptions apply. Therefore, the claim's ce

In [29]:
print("LANGUAGE:")
print(result_en.language)

print("\nML SIGNAL:")
print(result_en.ml_signal)

print("\nFINAL VERDICT:")
print(result_en.final_verdict.verdict)

print("\nFINAL EXPLANATION:")
print(result_en.final_verdict.explanation)

LANGUAGE:
en

ML SIGNAL:
{'prediction': 'Fake', 'label': 0, 'decision_score': -0.07834686387982306}

FINAL VERDICT:
REFUTED

FINAL EXPLANATION:
The article’s central claims are refuted by sufficient evidence. EU anti-money-laundering rules due to apply from 10 July 2027 set a €10,000 limit on certain cash transactions; they do not abolish cash payments. Cash payments below that limit remain possible, subject to the scope and exceptions of the rules. Consequently, citizens and businesses will not be required to use digital payments exclusively from July 2027. The auxiliary ML signal also labels the article as fake, but the verdict is based primarily on the sufficient claim-level evidence.


##### 5.1.1. Resultado de la integración ML

La noticia en inglés es clasificada por el modelo ML como `Fake`, con un `decision_score` de aproximadamente `-0.078`.

Este valor se encuentra próximo a la frontera de decisión del clasificador, por lo que la predicción ML puede considerarse poco marcada en términos de distancia al hiperplano. No obstante, el `decision_score` no representa una probabilidad ni una medida calibrada de confianza.

El Verdict Synthesizer devuelve finalmente `REFUTED`, apoyándose principalmente en la evidencia factual suficiente obtenida para las claims. La señal ML es coherente con el resultado factual, pero actúa únicamente como información auxiliar.

### 6. Conclusión de la integración end-to-end

Las pruebas realizadas permiten validar el funcionamiento conjunto de los principales componentes del sistema.

El backend integra correctamente la detección de idioma, extracción de claims, investigación web, recuperación de evidencias, verificación factual, señal auxiliar de Machine Learning para noticias en inglés y síntesis del veredicto final.

La salida estructurada conserva además la trazabilidad necesaria para inspeccionar las claims extraídas, las evidencias utilizadas, la valoración de cada evidencia y el resultado final de la noticia.

Con esta integración funcional validada, el siguiente paso consiste en representar y orquestar este flujo mediante LangGraph, manteniendo la lógica de los componentes ya desarrollados.

# LangGraph

### 7. Validación del flujo con LangGraph

Una vez `definidos` el estado, los nodos y las conexiones del grafo (en graph.py), se valida la ejecución completa del sistema mediante LangGraph.

El objetivo es comprobar que la orquestación reproduce correctamente el comportamiento previamente validado mediante el backend secuencial.

En primer lugar, se utiliza una noticia en español para verificar que:

- Se detecta correctamente el idioma;
- Se extraen las claims;
- Se ejecuta el flujo de verificación factual;
- La rama de Machine Learning no se ejecuta;
- El Verdict Synthesizer genera el veredicto final.

In [49]:
import importlib
import src.graph as graph_module

importlib.reload(graph_module)

from src.graph import build_news_graph

In [32]:
title = "La Unión Europea prohibirá completamente los pagos en efectivo en 2027"

body = """
La Unión Europea eliminará por completo la posibilidad de realizar pagos
en efectivo a partir de 2027. La medida entrará en vigor en julio de ese año
y obligará a ciudadanos y empresas a utilizar exclusivamente medios de pago
digitales.
"""

In [33]:
news_graph = build_news_graph(
    client=client,
    tavily_client=tavily_client,
    embedding_model=embedding_model,
)

In [41]:
input_es = {
    "title": title,
    "body": body,
}

In [42]:
final_state_es = input_es.copy()

for event in news_graph.stream(
    input_es,
    stream_mode="updates",
):

    for node_name, update in event.items():

        print(f"NODE: {node_name}")

        final_state_es.update(update)

NODE: detect_language
NODE: claim_analyzer
NODE: factual_verification
NODE: verdict_synthesizer


In [43]:
print("\nLANGUAGE:")
print(final_state_es["language"])

print("\nNUMBER OF CLAIMS:")
print(len(final_state_es["claims"]))

print("\nML SIGNAL:")
print(final_state_es.get("ml_signal"))

print("\nFINAL VERDICT:")
print(final_state_es["final_verdict"].verdict)

print("\nFINAL EXPLANATION:")
print(final_state_es["final_verdict"].explanation)


LANGUAGE:
es

NUMBER OF CLAIMS:
2

ML SIGNAL:
None

FINAL VERDICT:
REFUTED

FINAL EXPLANATION:
El artículo queda refutado. Las dos afirmaciones centrales son falsas según evidencia suficiente: desde el 10 de julio de 2027 la UE no prohibirá por completo los pagos en efectivo ni impondrá el uso exclusivo de medios digitales. La normativa establece un límite máximo de 10.000 euros para determinados pagos en efectivo, por lo que los pagos dentro de ese límite seguirán siendo posibles.


##### 7.1.1. Resultado de la validación en español

La ejecución mediante LangGraph detecta correctamente el idioma español y extrae dos claims verificables de la noticia.

El flujo ejecuta la rama de verificación factual y omite correctamente el nodo correspondiente al clasificador de Machine Learning, por lo que `ml_signal` permanece como `None`.

El Verdict Synthesizer genera finalmente un resultado `REFUTED` y produce la explicación en el idioma original de la noticia.

Esta prueba confirma que la ruta correspondiente a noticias en español se ejecuta correctamente dentro del grafo y reproduce el comportamiento previamente validado mediante el backend secuencial.

### 7.2. Validación del flujo para noticias en inglés

Se realiza una segunda ejecución utilizando una noticia en inglés. En este caso, además del flujo de verificación factual, el grafo `debe ejecutar la rama correspondiente al clasificador de Machine Learning`.

El objetivo es comprobar que la condición basada en el idioma activa correctamente el nodo ML y que su salida se incorpora posteriormente al Verdict Synthesizer.

In [44]:
title_en = (
    "The European Union will completely ban cash payments from 2027"
)

body_en = """
The European Union will completely eliminate cash payments starting
in July 2027. Citizens and businesses across the EU will be required
to use digital payment methods exclusively.
"""

In [45]:
input_en = {
    "title": title_en,
    "body": body_en,
}

In [46]:
final_state_en = input_en.copy()

for event in news_graph.stream(
    input_en,
    stream_mode="updates",
):
    for node_name, update in event.items():

        print(f"NODE: {node_name}")

        final_state_en.update(update)

NODE: detect_language
NODE: claim_analyzer
NODE: factual_verification
NODE: ml
NODE: verdict_synthesizer


In [47]:
print("\nLANGUAGE:")
print(final_state_en["language"])

print("\nNUMBER OF CLAIMS:")
print(len(final_state_en["claims"]))

print("\nML SIGNAL:")
print(final_state_en.get("ml_signal"))

print("\nFINAL VERDICT:")
print(final_state_en["final_verdict"].verdict)

print("\nFINAL EXPLANATION:")
print(final_state_en["final_verdict"].explanation)


LANGUAGE:
en

NUMBER OF CLAIMS:
2

ML SIGNAL:
{'prediction': 'Fake', 'label': 0, 'decision_score': -0.07834686387982306}

FINAL VERDICT:
REFUTED

FINAL EXPLANATION:
The article’s central claims are refuted by sufficient evidence. The EU is not abolishing cash or requiring exclusively digital payments from July 2027. That date relates to the application of anti-money-laundering rules imposing a €10,000 cap on certain cash transactions, while cash remains legal tender and private non-professional transactions retain exceptions. The auxiliary ML signal also classifies the article as fake, consistent with the claim-level findings.


#### 7.2.1. Resultado de la validación en inglés

La ejecución mediante LangGraph detecta correctamente el idioma inglés y extrae dos claims verificables.

A diferencia de la ruta en español, el grafo ejecuta también el nodo correspondiente al clasificador de Machine Learning. La señal obtenida se conserva dentro del estado del grafo mediante `ml_signal`.

El clasificador devuelve una predicción `Fake` junto con su `decision_score`. Este valor se conserva únicamente como información técnica y no se interpreta como una probabilidad de veracidad.

El Verdict Synthesizer genera finalmente un resultado `REFUTED`. La explicación prioriza los resultados de verificación factual y utiliza la predicción del modelo ML únicamente como señal auxiliar.

Esta prueba confirma que la ruta condicional del grafo funciona correctamente para noticias en inglés y que la señal ML se integra de acuerdo con la jerarquía definida para el sistema.

### 8. Refinamiento de la arquitectura LangGraph

Tras validar el funcionamiento completo del grafo para noticias en español e inglés, se refina la estructura de orquestación para representar de forma más explícita las distintas etapas del proceso de verificación factual.

En la primera implementación, el Research Agent, el Live RAG y el Evidence Verifier se encontraban agrupados dentro de un único nodo denominado `factual_verification`. Esta decisión permitió validar inicialmente que LangGraph reproducía correctamente el comportamiento del backend secuencial.

Una vez validado el flujo, estos componentes se separan en tres nodos independientes:

- `research_agent`: realiza la búsqueda y selección de documentos candidatos para cada claim.
- `live_rag`: procesa los documentos y recupera los fragmentos de evidencia más relevantes.
- `evidence_verifier`: determina la relación de las evidencias con cada claim y genera su veredicto factual.

Esta modificación no altera la lógica interna de los componentes previamente desarrollados, sino únicamente su representación y orquestación dentro de LangGraph.

La separación permite que el grafo refleje con mayor fidelidad la arquitectura del sistema, mejora la trazabilidad de las diferentes etapas y facilitará posteriormente la visualización del proceso de análisis en la interfaz desarrollada con Streamlit.

#### 8.1. Validación del grafo refinado

Tras separar el nodo de verificación factual en tres componentes independientes (`research_agent`, `live_rag` y `evidence_verifier`), se realizan nuevas pruebas para comprobar que la modificación afecta únicamente a la orquestación y no altera la lógica funcional previamente validada.

El objetivo principal es verificar que LangGraph ejecuta correctamente cada etapa del pipeline como un nodo independiente y que mantiene el routing condicional según el idioma de la noticia.

Para noticias en español, el flujo esperado es:

`detect_language → claim_analyzer → research_agent → live_rag → evidence_verifier → verdict_synthesizer`

sin ejecutar el nodo de Machine Learning.

Para noticias en inglés, el flujo esperado es:

`detect_language → claim_analyzer → research_agent → live_rag → evidence_verifier → ml → verdict_synthesizer`.

Estas pruebas permiten validar que la representación del grafo refleja correctamente la arquitectura modular del sistema y conserva el comportamiento previamente observado en el backend secuencial.

In [50]:
news_graph = build_news_graph(
    client=client,
    tavily_client=tavily_client,
    embedding_model=embedding_model,
)

In [51]:
final_state_es = input_es.copy()

for event in news_graph.stream(
    input_es,
    stream_mode="updates",
):
    for node_name, update in event.items():

        print(f"NODE: {node_name}")

        final_state_es.update(update)

NODE: detect_language
NODE: claim_analyzer
NODE: research_agent
NODE: live_rag
NODE: evidence_verifier
NODE: verdict_synthesizer


In [52]:
final_state_en = input_en.copy()

for event in news_graph.stream(
    input_en,
    stream_mode="updates",
):
    for node_name, update in event.items():

        print(f"NODE: {node_name}")

        final_state_en.update(update)

NODE: detect_language
NODE: claim_analyzer
NODE: research_agent
NODE: live_rag
NODE: evidence_verifier
NODE: ml
NODE: verdict_synthesizer


#### 8.1.1. Resultado de la validación

Las ejecuciones confirman que el grafo refinado mantiene correctamente el comportamiento esperado.

Para noticias en español, LangGraph ejecuta los nodos de detección de idioma, extracción de claims, investigación, recuperación de evidencias, verificación factual y síntesis final, omitiendo correctamente el nodo de Machine Learning.

Para noticias en inglés, el mismo flujo incorpora adicionalmente el nodo `ml` antes del Verdict Synthesizer.

Estos resultados verifican que la separación del antiguo nodo `factual_verification` en `research_agent`, `live_rag` y `evidence_verifier` no altera la lógica funcional del sistema y permite representar de forma más explícita y trazable las distintas etapas del proceso de verificación.